In [30]:
import matplotlib.pyplot as plt
import numpy as np
from ham import *

## OSL Calibration (Open-Short-Load) - NanoVNA1

### What is OSL Calibration?

OSL calibration corrects **systematic errors** in VNA measurements by measuring three known standards and solving for error terms.

### Three Systematic Errors in 1-Port Measurements:

1. **Directivity Error (Ed)** - Leakage from source to reflected signal path (crosstalk)
2. **Source Match Error (Es)** - Reflections from imperfect 50Ω source impedance
3. **Reflection Tracking Error (Et)** - Frequency response of forward and reverse paths

### The Three Calibration Standards:

| Standard | Γ (Reflection Coeff) | Purpose |
|----------|---------------------|----------|
| **SHORT** | -1 ∠180° | Perfect reflection, negative |
| **OPEN** | +1 ∠0° | Perfect reflection, positive |
| **LOAD** | 0 ∠0° | Perfect match (no reflection) |

### The Math: Error Box Model

For any measurement, the **raw measured** reflection coefficient Γm is related to the **actual** Γa by:

```
Γm = Ed + (Er × Γa) / (1 - Es × Γa)
```

Where:
- **Ed** = Directivity error
- **Es** = Source match error  
- **Er** = Reflection tracking error (often written as Et)

### How OSL Calibration Works:

1. **Measure SHORT**: Γa = -1, get Γm_short
2. **Measure OPEN**: Γa = +1, get Γm_open
3. **Measure LOAD**: Γa = 0, get Γm_load

This gives 3 equations with 3 unknowns (Ed, Es, Er), which can be solved!

### After Calibration:

Once Ed, Es, Er are known, any measured Γm can be corrected to actual Γa:

```
Γa = (Γm - Ed) / (Er + Es × (Γm - Ed))
```

### Why These Three Standards?

- **LOAD** directly reveals **Ed** (directivity) since Γa=0 means Γm_load = Ed
- **SHORT** and **OPEN** are opposite extremes (±1), giving maximum leverage to solve for **Es** and **Er**
- Three standards = three equations to solve for three unknowns

### Practical Importance:

Without calibration, measurements include:
- ❌ Cable reflections
- ❌ Connector reflections  
- ❌ Frequency response errors
- ❌ Directional coupler imperfections

After OSL calibration:
- ✅ Reference plane moved to calibration point
- ✅ Systematic errors removed
- ✅ Accurate impedance and VSWR readings

In [34]:
# Demonstrate OSL Calibration Process
print("="*80)
print("OSL CALIBRATION SIMULATION - How NanoVNA1 Removes Systematic Errors")
print("="*80)

# Simulate systematic errors in the VNA
Ed = 0.05 * np.exp(1j * np.deg2rad(30))   # Directivity error: 5% @ 30°
Es = 0.03 * np.exp(1j * np.deg2rad(-45))  # Source match error: 3% @ -45°
Er = 0.98 * np.exp(1j * np.deg2rad(5))    # Reflection tracking: 0.98 @ 5°

print(f"\nSimulated VNA Errors:")
print(f"  Directivity (Ed):       |{abs(Ed):.4f}| ∠{np.degrees(np.angle(Ed)):6.1f}°")
print(f"  Source Match (Es):      |{abs(Es):.4f}| ∠{np.degrees(np.angle(Es)):6.1f}°")
print(f"  Reflection Tracking(Er):|{abs(Er):.4f}| ∠{np.degrees(np.angle(Er)):6.1f}°")

# Function to simulate VNA measurement with errors
def vna_measure_raw(Gamma_actual, Ed, Es, Er):
    """Simulate what VNA measures (with errors)"""
    return Ed + (Er * Gamma_actual) / (1 - Es * Gamma_actual)

# STEP 1: Perform OSL Calibration - Measure the three standards
print(f"\n{'='*80}")
print("STEP 1: CALIBRATION - Measure Known Standards")
print("="*80)

# Known standard values
Gamma_short = -1.0 + 0j  # Perfect short
Gamma_open = +1.0 + 0j   # Perfect open
Gamma_load = 0.0 + 0j    # Perfect 50Ω load

# Measure standards (raw measurements include errors)
Gamma_m_short = vna_measure_raw(Gamma_short, Ed, Es, Er)
Gamma_m_open = vna_measure_raw(Gamma_open, Ed, Es, Er)
Gamma_m_load = vna_measure_raw(Gamma_load, Ed, Es, Er)

print(f"\n{'Standard':<15} {'Actual Γ':<25} {'Measured Γ (with errors)':<25}")
print("-"*80)
print(f"{'SHORT':<15} {Gamma_short.real:7.4f} + j{Gamma_short.imag:7.4f} ({abs(Gamma_short):.3f}∠{np.degrees(np.angle(Gamma_short)):6.1f}°) ", end="")
print(f"{Gamma_m_short.real:7.4f} + j{Gamma_m_short.imag:7.4f} ({abs(Gamma_m_short):.3f}∠{np.degrees(np.angle(Gamma_m_short)):6.1f}°)")

print(f"{'OPEN':<15} {Gamma_open.real:7.4f} + j{Gamma_open.imag:7.4f} ({abs(Gamma_open):.3f}∠{np.degrees(np.angle(Gamma_open)):6.1f}°) ", end="")
print(f"{Gamma_m_open.real:7.4f} + j{Gamma_m_open.imag:7.4f} ({abs(Gamma_m_open):.3f}∠{np.degrees(np.angle(Gamma_m_open)):6.1f}°)")

print(f"{'LOAD (50Ω)':<15} {Gamma_load.real:7.4f} + j{Gamma_load.imag:7.4f} ({abs(Gamma_load):.3f}∠{np.degrees(np.angle(Gamma_load)):6.1f}°) ", end="")
print(f"{Gamma_m_load.real:7.4f} + j{Gamma_m_load.imag:7.4f} ({abs(Gamma_m_load):.3f}∠{np.degrees(np.angle(Gamma_m_load)):6.1f}°)")

# STEP 2: Solve for error terms
print(f"\n{'='*80}")
print("STEP 2: SOLVE for Error Terms")
print("="*80)

# From the three measurements, we can solve for Ed, Es, Er
# Using the simplified OSL equations:

# STEP 2: Solve for error terms using CORRECT OSL formulas
print(f"\n{'='*80}")
print("STEP 2: SOLVE for Error Terms (CORRECT FORMULAS)")
print("="*80)

# From LOAD measurement (ra = 0):
# ra = (rm - Ed_calc) / (Er_calc + Es_calc * (rm - Ed_calc))
# 0 = (rm - Ed_calc)
# => Ed_calc = rm
Ed_calc = Gamma_m_load

# From SHORT and OPEN measurements:
# Correct formula: Es = (Γm_short + Γm_open - 2*Ed) / (Γm_open - Γm_short)
Es_calc = (Gamma_m_short + Gamma_m_open - 2*Ed_calc) / (Gamma_m_open - Gamma_m_short)

# Correct formula: Er = -(Γm_short - Ed) * (1 + Es)
# Or equivalently: Er = (Γm_open - Ed) * (1 - Es)
Er_calc = -(Gamma_m_short - Ed_calc) * (1 + Es_calc)

print(f"\nCalculated Error Terms (from calibration):")
print(f"  Ed_calc: |{abs(Ed_calc):.4f}| ∠{np.degrees(np.angle(Ed_calc)):6.1f}° ", end="")
print(f"(actual: |{abs(Ed):.4f}| ∠{np.degrees(np.angle(Ed)):6.1f}°) ✓")

print(f"  Es_calc: |{abs(Es_calc):.4f}| ∠{np.degrees(np.angle(Es_calc)):6.1f}° ", end="")
print(f"(actual: |{abs(Es):.4f}| ∠{np.degrees(np.angle(Es)):6.1f}°) ✓")

print(f"  Er_calc: |{abs(Er_calc):.4f}| ∠{np.degrees(np.angle(Er_calc)):6.1f}° ", end="")
print(f"(actual: |{abs(Er):.4f}| ∠{np.degrees(np.angle(Er)):6.1f}°) ✓")

# STEP 3: Test calibration with unknown DUT
print(f"\n{'='*80}")
print("STEP 3: MEASURE Unknown Antenna (DUT)")
print("="*80)

# Simulate measuring a real antenna: 75Ω + j30Ω
Z_antenna = 75 + 30j
Gamma_antenna_actual = impedance_to_gamma(Z_antenna, 50)
VSWR_antenna_actual = (1 + abs(Gamma_antenna_actual)) / (1 - abs(Gamma_antenna_actual))

# Raw measurement (with errors)
Gamma_antenna_raw = vna_measure_raw(Gamma_antenna_actual, Ed, Es, Er)

# Apply calibration correction
def apply_calibration(Gamma_measured, Ed_calc, Es_calc, Er_calc):
    """Apply OSL calibration to correct measurement"""
    return (Gamma_measured - Ed_calc) / (Er_calc + Es_calc * (Gamma_measured - Ed_calc))

Gamma_antenna_corrected = apply_calibration(Gamma_antenna_raw, Ed_calc, Es_calc, Er_calc)

# Calculate impedances
Z_raw = gamma_to_impedance(Gamma_antenna_raw, 50)
Z_corrected = gamma_to_impedance(Gamma_antenna_corrected, 50)
VSWR_raw = (1 + abs(Gamma_antenna_raw)) / (1 - abs(Gamma_antenna_raw))
VSWR_corrected = (1 + abs(Gamma_antenna_corrected)) / (1 - abs(Gamma_antenna_corrected))

# print(f"\nAntenna: Z = {Z_antenna.real:.1f} + j{Z_antenna.imag:.1f} Ω")
# print(f"Expected VSWR: {VSWR_antenna_actual:.3f}")
# print(f"\n{'Measurement':<25} {'Z (Ω)':<25} {'VSWR':<12} {'Error'}")
# print("-"*80)
# print(f"{'Actual (True Value)':<25} {Z_antenna.real:6.1f} + j{Z_antenna.imag:6.1f}           {VSWR_antenna_actual:<12.3f} {'—'}")
print(f"\nAntenna: Z = {Z_antenna.real:.1f} + j{Z_antenna.imag:.1f} Ω")
print(f"Expected VSWR: {VSWR_antenna_actual:.3f}")
print(f"\n{'Measurement':<25} {'Z (Ω)':<25} {'VSWR':<12} {'Error'}")
print("-"*80)
print(f"{'Actual (True Value)':<25} {Z_antenna.real:6.1f} + j{Z_antenna.imag:6.1f}           {VSWR_antenna_actual:<12.3f} {'—'}")
print(f"{'Raw (Uncalibrated)':<25} {Z_raw.real:6.1f} + j{Z_raw.imag:6.1f}           {VSWR_raw:<12.3f} {'❌ WRONG'}")
print(f"{'Corrected (OSL Cal)':<25} {Z_corrected.real:6.1f} + j{Z_corrected.imag:6.1f}           {VSWR_corrected:<12.3f} {'✅ Perfect!'}")
# print(f"  • Impedance error removed: {abs(Z_corrected - Z_antenna):.2f}Ω")
print(f"\n{'='*80}")
# print("SUMMARY:")print(f"  • Without calibration: Z = {Z_raw.real:.1f}+j{Z_raw.imag:.1f}Ω, VSWR = {VSWR_raw:.2f}")print(f"  • With OSL calibration: Z = {Z_corrected.real:.1f}+j{Z_corrected.imag:.1f}Ω, VSWR = {VSWR_corrected:.2f}")print(f"  • Impedance error removed: {abs(Z_corrected - Z_antenna):.2f}Ω")print(f"  • This is why you MUST calibrate your NanoVNA before measurements!")print("="*80)

OSL CALIBRATION SIMULATION - How NanoVNA1 Removes Systematic Errors

Simulated VNA Errors:
  Directivity (Ed):       |0.0500| ∠  30.0°
  Source Match (Es):      |0.0300| ∠ -45.0°
  Reflection Tracking(Er):|0.9800| ∠   5.0°

STEP 1: CALIBRATION - Measure Known Standards

Standard        Actual Γ                  Measured Γ (with errors) 
--------------------------------------------------------------------------------
SHORT           -1.0000 + j 0.0000 (1.000∠ 180.0°) -0.9105 + j-0.0785 (0.914∠-175.1°)
OPEN             1.0000 + j 0.0000 (1.000∠   0.0°)  1.0422 + j 0.0906 (1.046∠   5.0°)
LOAD (50Ω)       0.0000 + j 0.0000 (0.000∠   0.0°)  0.0433 + j 0.0250 (0.050∠  30.0°)

STEP 2: SOLVE for Error Terms

STEP 2: SOLVE for Error Terms (CORRECT FORMULAS)

Calculated Error Terms (from calibration):
  Ed_calc: |0.0500| ∠  30.0° (actual: |0.0500| ∠  30.0°) ✓
  Es_calc: |0.0300| ∠ -45.0° (actual: |0.0300| ∠ -45.0°) ✓
  Er_calc: |0.9800| ∠   5.0° (actual: |0.9800| ∠   5.0°) ✓

STEP 3: MEASURE Unk

In [ ]:

Ztest = 49 + 1j  # Slightly off from 50Ω
print(f"\nTesting impedance to gamma conversion:", f"Z = {Ztest.real:.1f} + j{Ztest.imag:.1f} Ω")
gt = impedance_to_gamma(Ztest, 50)
print(f"Calculated Γ: {gt.real:.4f} + {gt.imag:.4f}j (|Γ|={abs(gt):.4f}, phase = {np.degrees(np.angle(gt)):.1f}°)")
#gamma = impedance_to_gamma(Ztest, 50)
#print(f"Impedance: {Ztest.real:.1f} + j{Ztest.imag:.1f} Ω → Γ = {gamma.real:.4f} + j{gamma.imag:.4f} (|Γ|={abs(gamma):.4f})")


Testing impedance to gamma conversion: Z = 49.0 + j1.0 Ω
Calculated Γ: -0.0100 + 0.0102j (|Γ|=0.0143, phase = 134.4°)
